# 🤖 Chapter 14: MLOps and Machine Learning Engineering

📖 Book: Chapter 14, pp. 574–619 · ⏱️ 2–3 hours · 🧱 Needs: Ch 11–12 · Level: Advanced

---

> **🎯 Key idea:** Treat ML models as versioned microservices: prevent train/serve skew, track runs with MLflow, and never deserialize untrusted `joblib`/`pickle` files.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Ad-hoc `.joblib` files on disk  
> **🆕 Now (2026):** MLflow tracking + versioned model registry  
> **Why:** Reproducible metrics, params & rollbacks

> **📜 Then (book, ~2020):** Unchecked `joblib.load()`  
> **🆕 Now (2026):** Trusted registries only or `ONNX`/`safetensors`  
> **Why:** `pickle`/`joblib` execute arbitrary code

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A data scientist hands you a Jupyter notebook that predicts housing prices, and asks you to put it in production by tomorrow. If scaling differs between training and inference, or bad JSON hits `/predict`, predictions silently break or crash. MLOps bridges training, evaluation, safe serialization, and containerized serving.

### Concept: Train/Serve Consistency
**What:** Identical preprocessing in train & `/predict`  
**Why:** Fitting scalers on full data causes leakage  
**How:** Fit `StandardScaler` on `X_train` only; save alongside model  
**When:** Every supervised ML pipeline

📖 *Book listing:* Height/Weight 80/20 Regression & RMSE (pp. 578–592)

**Plain-language analogy:** Train/serve skew is like studying metric units in school and taking the exam in imperial units—the scaler fit on `X_train` must be reused identically at `/predict`.

**Walkthrough:** The book splits data 80/20, fits a linear model on training data only to prevent leakage, and evaluates predictions on the held-out test set using RMSE.

### Concept: Model Serving & Drift
**What:** Flask `/predict` + RMSE + drift checks  
**Why:** Prod data shifts degrade accuracy silently  
**How:** Validate JSON schema (`400`) & alert on mean shift >20%  
**When:** All production ML endpoints

📖 *Book listing:* `GridSearchCV` + `joblib` + Flask `/predict` (pp. 605–617)

**Plain-language analogy:** Loading an untrusted `.joblib`/`.pkl` file is like running an unknown `.exe` from email—pickle can execute arbitrary code when opened.

**Walkthrough:** The book tunes `GradientBoostingRegressor`, exports it with `joblib.dump()`, and serves a `/predict` POST JSON endpoint in a Docker/Kubernetes microservice.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Pure-Python 80/20 split, linear regression, and RMSE
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Pure-Python 80/20 split, linear regression, and RMSE
import math

heights = [60, 62, 65, 68, 70, 72, 74, 75, 64, 69]
weights = [115, 122, 138, 152, 165, 178, 190, 195, 132, 158]
split = int(len(heights) * 0.8)
X_train, X_test = heights[:split], heights[split:]
y_train, y_test = weights[:split], weights[split:]

mx, my = sum(X_train) / len(X_train), sum(y_train) / len(y_train)
slope = sum((x - mx) * (y - my) for x, y in zip(X_train, y_train)) / sum((x - mx) ** 2 for x in X_train)
intercept = my - slope * mx
preds = [slope * x + intercept for x in X_test]
rmse = math.sqrt(sum((p - y) ** 2 for p, y in zip(preds, y_test)) / len(y_test))
print(f'y = {slope:.2f}x + ({intercept:.2f}) | Test RMSE: {rmse:.2f} lbs')

y = 5.47x + (-216.26) | Test RMSE: 2.40 lbs


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Min-Max feature scaler
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Min-Max feature scaler
def min_max_scale(values: list[float]) -> list[float]:
    mn, mx = min(values), max(values)
    return [0.0 if mx == mn else (v - mn) / (mx - mn) for v in values]

print('Scaled:', min_max_scale([10.0, 20.0, 30.0]))

Scaled: [0.0, 0.5, 1.0]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Model registry metadata manifest
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Model registry metadata manifest
import json
def build_model_manifest(name: str, version: str, features: list[str], rmse_val: float) -> dict:
    return {'model': name, 'version': version, 'features': features, 'metrics': {'rmse': round(rmse_val, 4)}}

manifest = build_model_manifest('weight-predictor', '1.0.0', ['height_in'], rmse)
print(json.dumps(manifest))

{"model": "weight-predictor", "version": "1.0.0", "features": ["height_in"], "metrics": {"rmse": 2.4021}}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: `/predict` Endpoint Validator & `sklearn` / MLflow

#### Example 4: Intermediate Example 1: /predict JSON schema validator
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: /predict JSON schema validator
def predict_endpoint(payload: dict, slope_val: float, intercept_val: float) -> tuple[int, dict]:
    features = payload.get('heights')
    if not isinstance(features, list) or not features or not all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in features):
        return 400, {'error': 'heights must be a non-empty list of numbers'}
    return 200, {'predictions': [round(slope_val * x + intercept_val, 2) for x in features]}

print('Valid:', predict_endpoint({'heights': [66, 71]}, slope, intercept))
print('Invalid:', predict_endpoint({'heights': []}, slope, intercept))

Valid: (200, {'predictions': [144.57, 171.91]})
Invalid: (400, {'error': 'heights must be a non-empty list of numbers'})


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): scikit-learn + MLflow
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): scikit-learn + MLflow
# Requires: pip install scikit-learn joblib mlflow flask
# SECURITY WARNING: Never call joblib.load() or pickle.load() on untrusted files!
try:
    import sklearn
    print('scikit-learn installed:', sklearn.__version__)
except ImportError:
    print('scikit-learn/mlflow not installed; simulated MLflow run:', {'run_id': 'demo-001', 'rmse': round(rmse, 4)})

scikit-learn/mlflow not installed; simulated MLflow run: {'run_id': 'demo-001', 'rmse': 2.4021}


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Feature Drift Detector** alerting when production mean shifts >20% from training baseline.

#### Example 6: def detect_feature_drift(train_mean: float, prod_mean: float, threshold: float = 0.20) -> bool:
*Expected output:* Printed labelled output below.

In [6]:
def detect_feature_drift(train_mean: float, prod_mean: float, threshold: float = 0.20) -> bool:
    return abs(prod_mean - train_mean) / abs(train_mean) > threshold

print('Drift detected (68 -> 85):', detect_feature_drift(68.0, 85.0))

Drift detected (68 -> 85): True


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: subprocess import guard for sklearn
import shutil, subprocess
py = shutil.which('python3')
if py:
    cp = subprocess.run([py, '-c', 'import sklearn; print(sklearn.__version__)'], capture_output=True, text=True)
    print('sklearn in env:', cp.stdout.strip() if cp.returncode == 0 else 'not installed (using pure-Python pipeline)')

sklearn in env: not installed (using pure-Python pipeline)


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Fitting scaler before split** -> Inflated test accuracy (leakage). ✅ Split first, then fit on `X_train`.
- ❌ **Loading untrusted `.joblib`** -> Remote code execution risk. ✅ Only load signed/trusted artifacts.
- ❌ **No `/predict` input validation** -> `500` crash on bad JSON. ✅ Validate types & return `400`.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement compute_rmse and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement compute_rmse and run to verify!)
def _run_starter_checks():
    assert compute_rmse([10, 20], [11, 19]) == 1.0
def compute_rmse(y_true: list[float], y_pred: list[float]) -> float:
    raise NotImplementedError('Implement compute_rmse')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement compute_rmse


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why must `StandardScaler` be fit only on `X_train`?

<details>
<summary>✅ Answer 1</summary>

Fitting on `X_test` leaks test distribution info into training.

</details>

**Q2:** How do `scikit-learn` and `PyTorch` differ?

<details>
<summary>✅ Answer 2</summary>

`scikit-learn` targets tabular CPU ML; `PyTorch` builds GPU-accelerated neural networks.

</details>

**Q3:** Why is `pickle`/`joblib` dangerous on untrusted files?

<details>
<summary>✅ Answer 3</summary>

Deserializing untrusted pickle streams can execute arbitrary Python code.

</details>

**Q4:** What does MLflow add to `joblib` + Flask?

<details>
<summary>✅ Answer 4</summary>

Experiment parameter/metric tracking and a versioned model registry.

</details>

**Q5:** Why does EDA matter before modeling?

<details>
<summary>✅ Answer 5</summary>

Catches missing values, outliers, and leakage before wasting training compute.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 3**: `sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import math

def compute_rmse(y_true: list[float], y_pred: list[float]) -> float:
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true))

def has_missing_values(rows: list[dict]) -> bool:
    return any(v is None for r in rows for v in r.values())

def compute_mae(y_true: list[float], y_pred: list[float]) -> float:
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)

def train_val_test_split(data: list, train_pct: float = 0.7, val_pct: float = 0.15) -> tuple:
    i1 = int(len(data) * train_pct)
    i2 = i1 + int(len(data) * val_pct)
    return data[:i1], data[i1:i2], data[i2:]

def deploy_model_gate(manifest: dict, max_rmse: float = 5.0) -> dict:
    r = manifest.get('metrics', {}).get('rmse')
    return {'promote': r is not None and r <= max_rmse and bool(manifest.get('features')), 'rmse': r}

assert compute_rmse([10, 20], [11, 19]) == 1.0
assert has_missing_values([{'a': 1}, {'a': None}]) is True
assert compute_mae([10, 20], [12, 18]) == 2.0
assert len(train_val_test_split(list(range(20)))[0]) == 14
assert deploy_model_gate(manifest)['promote'] is True
print('All Chapter 14 assertions PASSED!')

All Chapter 14 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `train_test_split(..., test_size=0.2)` — Holdout evaluation (e.g. Compute `RMSE` on test set)
- `joblib.dump(model, path)` — Serialize sklearn estimator (e.g. Load only from trusted registry)
- `POST /predict` — Serve JSON predictions (e.g. Validate schema -> return `200`/`400`)

### ✅ What You Can Now Do
- Implement 80/20 train/test splitting, linear regression, and RMSE evaluation without data leakage.
- Validate `/predict` JSON payloads (`400` vs `200`) and detect production feature drift.
- Track models with MLflow manifests and avoid untrusted `pickle`/`joblib` deserialization.

### ➡️ Next Chapter
Chapter 15 builds on this by engineering the scalable data pipelines that feed training and analytics.